# CORTEG — easy start

CORTEG adapts a **frozen EEG foundation model** (ST-EEGFormer, pre-trained on scalp EEG) to
**intracranial** ECoG decoding, training only ~297 K parameters: LoRA on the last four
transformer blocks, a spatial adapter, LayerNorms, and the readout head.

This notebook takes you from a fresh clone to reproducing a number from the paper.

## 1.0 Environment

| Package | Version | Note |
|---------|:-------:|------|
| torch   | 2.4+    | Deep learning framework |
| timm    | 1.0.10  | Provides the ViT blocks the backbone builds on |
| numpy   | 1.26+   | |
| scipy   | 1.11+   | Reads the electrode `.mat` files |

```bash
pip install -r requirements.txt
```

Everything below runs on CPU. A GPU only matters for section 5 (training).

Codes written by Liuyin Yang (liuyin.yang@kuleuven.be). All rights reserved.

---

## 1.1 The two checkpoints

CORTEG is **two files**, and this trips people up:

| File | Size | What it is | Where from |
|------|-----:|------------|------------|
| `checkpoint-300.pth` | 376 MB | the **frozen** ST-EEGFormer backbone | download once, see README |
| `checkpoints/corteg_stanford_pooled.pt` | 1.2 MB | the **trained** CORTEG adapter | ships in this repo |

The 1.2 MB file is small because it holds *only the adapted parameters*. The backbone is
frozen during training, so re-shipping it would be 376 MB of bytes identical to the download.

Get the backbone (once):

```bash
mkdir -p ~/checkpoints && cd ~/checkpoints
wget https://github.com/LiuyinYang1101/STEEGFormer/releases/download/ST-EEGFormer-small/checkpoint-300.pth
export CORTEG_PRETRAINED_ROOT=~/checkpoints
```

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # run from notebooks/

import numpy as np
import torch

import paths
from load_corteg import load_corteg, predict, verify_checkpoint

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("pretrained root:", paths.get_pretrained_root())
print("data root      :", paths.get_data_root())

torch 2.11.0+cu128 | cuda True
pretrained root: ~/workspace/datasets/pretrained_eeg_mae
data root      : ~/workspace/datasets/stanford_ecog


## 1.2 Check the released adapter before using it

The manifest records the file's SHA-256, its parameter count, and the exact architecture
arguments of the run that produced it. `verify_checkpoint` fails loudly if the file on disk
is not the file the paper used.

In [2]:
man = verify_checkpoint()

print(f"{man['name']}")
print(f"  {man['bytes']:,} bytes, sha256 {man['sha256'][:16]}…")
print(f"  {man['trainable_params']:,} trainable parameters")
print(f"  paper Table {man['paper']['table']}, {man['paper']['row']}: "
      f"{man['paper']['metric']} = {man['paper']['value']}")
print()
for k, v in man["architecture_args"].items():
    print(f"  {k:24s} {v}")

CORTEG (pooled Stanford)
  1,210,847 bytes, sha256 77d63a214e0e0f2c…
  297,236 trainable parameters
  paper Table 1, CORTEG (pooled): Pearson r = 0.554

  steegformer_variant      small
  merge_strategy           average
  stream                   both
  channel_adapter          knn_soft_fourier
  knn_k                    8
  adapter_branch           both
  xyz_mode                 real
  lora_r                   4
  lora_alpha               16
  lora_last_n              4
  lora_targets             qkv,proj,fc1,fc2
  hi_patch_size            25
  hi_inject_last_n         4
  use_ecog_fuser           True


## 2.0 Load the model

`load_corteg` rebuilds the architecture from the manifest, injects LoRA, materialises the
lazy head, and then loads the weights — checking that **every** tensor in the checkpoint
finds a home. If you rebuild a slightly different architecture it raises, rather than
loading nothing and reporting success.

Three arguments describe your recording:

- `C_in` — number of electrodes
- `T_in` — samples in the low-frequency window (128 for Stanford)
- `ecog_xyz_mm` — `(C_in, 3)` electrode coordinates **in millimetres**

In [3]:
C_in = 46                       # subject bp has 46 electrodes
xyz  = np.random.RandomState(0).randn(C_in, 3).astype(np.float32) * 30.0   # placeholder mm

model = load_corteg(C_in=C_in, T_in=128, ecog_xyz_mm=xyz, d_out=5, device="cpu")

n_all = sum(p.numel() for p in model.parameters())
print(f"total parameters     {n_all:,}")
print(f"released (trainable) {man['trainable_params']:,}"
      f"  ({100*man['trainable_params']/n_all:.1f}% of the model)")

[pretrained] path=~/workspace/datasets/pretrained_eeg_mae/experiment3_small/checkpoint-300.pth


[pretrained] strict=False missing=3 unexpected=0


[pretrained] --- missing keys (showing up to 80) ---


  MISSING: patch_embed_hi.proj.weight


  MISSING: patch_embed_hi.proj.bias


  MISSING: enc_time.pe


  channel_adapter=knn_soft_fourier attached (M=145, hidden=128)


total parameters     25,592,596
released (trainable) 297,236  (1.2% of the model)


### Reading the load messages

The `[pretrained] strict=False missing=3` line above is **expected, not a failure**. Three
tensors legitimately have no counterpart in a scalp-EEG checkpoint:

- `patch_embed_hi.proj.{weight,bias}` — the high-gamma patch embedder. It does not exist in
  the pre-trained model, so it is *warm-started* by interpolating the low-frequency patch
  embedder from 16 to 25 samples (you will see the `hi patch embed warm-started` line).
- `enc_time.pe` — the temporal positional encoding, rebuilt for this window length.

For the gated variant you will additionally see six `layerwise_gate.*` keys missing. Those
are the gate network, which is new and zero-initialised — that is exactly why the gate starts
at 0 in section 4.1.

A *long* list of missing keys, on the other hand, means the architecture does not match the
checkpoint. `load_corteg` raises in that case rather than letting it pass.


## 2.1 The two input streams

**This is the part to get right.** CORTEG is dual-stream. Both streams cover the same 1-second
window, but they are sampled differently:

| Stream | Shape | Content | Patch size | Tokens per electrode |
|--------|-------|---------|-----------:|---------------------:|
| `x_lo` | `(N, C, 128)` | broadband signal at 128 Hz | 16 | 128/16 = **8** |
| `x_hi` | `(N, C, 200)` | high-gamma envelope feature | 25 | 200/25 = **8** |

The patch sizes are chosen so both streams produce the **same token count**, which is what
lets them be fused. Passing only `x_lo` collapses the token count and the forward pass fails —
so `predict` refuses rather than silently giving you a different model.

In [4]:
N = 4
x_lo = np.random.randn(N, C_in, 128).astype(np.float32)
x_hi = np.random.randn(N, C_in, 200).astype(np.float32)

y = predict(model, x_lo, x_hi)
print("x_lo", x_lo.shape, "+ x_hi", x_hi.shape, "->", y.shape, "(one value per finger)")

try:
    predict(model, x_lo)                     # forgetting the hi stream
except ValueError as e:
    print("\nrefused, as intended:\n ", e)

x_lo (4, 46, 128) + x_hi (4, 46, 200) -> (4, 5) (one value per finger)

refused, as intended:
  this model was built with stream='both' and needs x_hi (the high-gamma stream); see data/stanford_preprocessing/


## 3.0 Reproduce a number from the paper

Now with real data. This needs the Stanford fingerflex dataset — see
`data/stanford_preprocessing/tutorial_Stanford.md` for the download and the
MATLAB + MNE pipeline that produces the `*_features.pkl` files.

Set `CORTEG_DATA_ROOT` to the directory holding them. The cell below skips itself if the
data is not present, so the notebook still runs end-to-end without it.

In [5]:
from data.io import load_subject
from data.scalers import fit_zscore_3d_per_channel, apply_zscore_3d_per_channel

SUBJECT = "bp"
root = paths.get_data_root()
have_data = os.path.exists(os.path.join(root, f"{SUBJECT}_features.pkl"))
print("Stanford data found." if have_data else f"No data at {root} — skipping sections 3.x")

if have_data:
    sd = load_subject(root, SUBJECT)
    print(f"\n{SUBJECT}: {sd.X_raw_te.shape[1]} electrodes, "
          f"{sd.X_raw_te.shape[0]} test windows")
    print(f"  x_lo  {sd.X_raw_te.shape}       broadband @128 Hz")
    print(f"  x_hi  {sd.X_feat_te[..., 0].shape}       high-gamma  (X_feat[..., 0])")
    print(f"  y     {sd.y_te.shape}         5 finger flexion traces")
    print(f"  xyz   {sd.ecog_xyz_mm.shape}           electrode coordinates in mm")

Stanford data found.



bp: 46 electrodes, 5225 test windows
  x_lo  (5225, 46, 128)       broadband @128 Hz
  x_hi  (5225, 46, 200)       high-gamma  (X_feat[..., 0])
  y     (5225, 5)         5 finger flexion traces
  xyz   (46, 3)           electrode coordinates in mm


## 3.1 Normalise exactly as training did

Both streams are z-scored **per channel**, with statistics fitted on the *training* split and
applied to the test split. Fitting on test would leak.

In [6]:
if have_data:
    raw_stats = fit_zscore_3d_per_channel(sd.X_raw_tr)
    hi_stats  = fit_zscore_3d_per_channel(sd.X_feat_tr[..., 0])

    x_lo_te = apply_zscore_3d_per_channel(sd.X_raw_te,          raw_stats)
    x_hi_te = apply_zscore_3d_per_channel(sd.X_feat_te[..., 0], hi_stats)
    print("normalised:", x_lo_te.shape, x_hi_te.shape)

normalised: (5225, 46, 128) (5225, 46, 200)


## 3.2 Predict, and compare against the paper

The metric is Pearson correlation between predicted and true flexion, computed per finger and
then averaged. The published per-subject value for `bp` is **0.5783**.

In [7]:
if have_data:
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    m = load_corteg(C_in=sd.X_raw_te.shape[1], T_in=x_lo_te.shape[-1],
                    ecog_xyz_mm=sd.ecog_xyz_mm, d_out=5, device=dev)
    y_pred = predict(m, x_lo_te, x_hi_te, batch_size=256)

    FINGERS = ["thumb", "index", "middle", "ring", "pinky"]
    r = [np.corrcoef(y_pred[:, i], sd.y_te[:, i])[0, 1] for i in range(5)]
    for name, v in zip(FINGERS, r):
        print(f"  {name:7s} r = {v:.4f}")
    print(f"\n  mean    r = {np.mean(r):.4f}      published: 0.5783")

[pretrained] path=~/workspace/datasets/pretrained_eeg_mae/experiment3_small/checkpoint-300.pth


[pretrained] strict=False missing=3 unexpected=0


[pretrained] --- missing keys (showing up to 80) ---


  MISSING: patch_embed_hi.proj.weight


  MISSING: patch_embed_hi.proj.bias


  MISSING: enc_time.pe


  channel_adapter=knn_soft_fourier attached (M=145, hidden=128)


  thumb   r = 0.6321
  index   r = 0.6503
  middle  r = 0.5063
  ring    r = 0.6129
  pinky   r = 0.4912

  mean    r = 0.5786      published: 0.5783


## 4.0 The two CORTEG variants

The paper reports two ways of fusing the high-gamma stream into the low-frequency stream:

| `--merge_strategy` | What it does |
|--------------------|--------------|
| `average` | **fixed fusion at layer k** — a 0.5/0.5 blend at one chosen depth. This is what the released checkpoint uses. |
| `layerwise_gate` | **gated fusion** — a small network sees the pooled initial tokens and emits one scalar per block; every block's output receives `+ g_l · hi`. |

The gate's last linear layer is zero-initialised, so with `tanh` every gate starts at exactly
**0** — the model begins as the low-frequency-only baseline and *learns* where to open the
high-gamma stream. Because `tanh` is signed, it can also learn to *suppress* it.

In [8]:
import json, types
from experiments.run_regression_hilo_clean import build_model, unfreeze_merge_params
from models.steegformer.probe import configure_lora_lastn_probe

base = json.load(open("../checkpoints/corteg_stanford_pooled.json"))["build_args"]

def build(strategy):
    a = dict(base)
    a.update(model_kwargs_json="../configs/steegformer_small.json",
             no_pretrained=False, merge_strategy=strategy)
    m = build_model(types.SimpleNamespace(**a), C_in=C_in, T_in=128,
                    ecog_xyz_m=xyz, d_out=5)
    configure_lora_lastn_probe(m, n_last=4, r=4, alpha=16, dropout=0.2,
                               targets=("qkv", "proj", "fc1", "fc2"))
    unfreeze_merge_params(m)
    return m

for s in ("average", "layerwise_gate"):
    m = build(s)
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"{s:16s} trainable = {n:,}")

[pretrained] path=~/workspace/datasets/pretrained_eeg_mae/experiment3_small/checkpoint-300.pth


[pretrained] strict=False missing=3 unexpected=0


[pretrained] --- missing keys (showing up to 80) ---


  MISSING: patch_embed_hi.proj.weight


  MISSING: patch_embed_hi.proj.bias


...

  MISSING: layerwise_gate.net.1.bias


  MISSING: layerwise_gate.net.3.weight


  MISSING: layerwise_gate.net.3.bias


  channel_adapter=knn_soft_fourier attached (M=145, hidden=128)


layerwise_gate   trainable = 313,255


## 4.1 Watch the gates open

Run a few optimisation steps on the gated variant and print the per-block gate values. They
start at exactly zero and move apart — some blocks let high-gamma in, others push it away.

In [9]:
gated = build("layerwise_gate")
xyz_t = torch.from_numpy(xyz / 1000.0).float().unsqueeze(0).expand(N, -1, -1)  # (B, C, 3) in metres
lo_t, hi_t = torch.from_numpy(x_lo), torch.from_numpy(x_hi)

def gates():
    gated(lo_t, x_hi=hi_t, ecog_xyz=xyz_t)
    return gated.backbone.last_aux["layerwise_gate_per_layer"].detach().numpy()

print("at init :", np.round(gates(), 3))

opt = torch.optim.AdamW([p for p in gated.parameters() if p.requires_grad], lr=1e-2)
target = torch.randn(N, 5)
for _ in range(5):
    loss = ((gated(lo_t, x_hi=hi_t, ecog_xyz=xyz_t) - target) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()

print("5 steps :", np.round(gates(), 3))
print("\n(positive = this block admits high-gamma, negative = suppresses it)")

[pretrained] path=~/workspace/datasets/pretrained_eeg_mae/experiment3_small/checkpoint-300.pth


[pretrained] strict=False missing=9 unexpected=0


[pretrained] --- missing keys (showing up to 80) ---


  MISSING: patch_embed_hi.proj.weight


  MISSING: patch_embed_hi.proj.bias


...

  MISSING: layerwise_gate.net.3.weight


  MISSING: layerwise_gate.net.3.bias


  channel_adapter=knn_soft_fourier attached (M=145, hidden=128)


at init : [0. 0. 0. 0. 0. 0. 0. 0.]


5 steps : [ 0.937  0.92   0.96   0.951  0.88  -0.119  0.397  0.203]

(positive = this block admits high-gamma, negative = suppresses it)


## 5.0 Train it yourself

The full training recipe behind the Table 1 row is one script:

```bash
bash scripts/table1_corteg_pooled_stanford.sh
```

Under the hood that is:

```bash
python -m experiments.run_regression_hilo_clean \
    --dataset Stanford --train_mode pooled \
    --steegformer_variant small \
    --model_kwargs_json configs/steegformer_small.json \
    --merge_strategy average \
    --channel_adapter knn_soft_fourier --knn_k 8 \
    --lora_last_n 4 --lora_r 4 --lora_alpha 16 \
    --epochs 100 --lr 3e-3 --weight_decay 0.005 --seed 42
```

For the **gated** variant, swap one flag:

```bash
    --merge_strategy layerwise_gate --layerwise_gate_act tanh
```

`--model_kwargs_json` is what loads the frozen backbone. Without it the runner now stops with
an error — an earlier release omitted it everywhere, which quietly trained a randomly
initialised backbone and reproduced the *random init* ablation instead of CORTEG.

For leave-one-subject-out fine-tuning: `bash scripts/table1_corteg_loo_ft.sh <subject>`.

## 6.0 Where to go next

- [`README.md`](../README.md) — install, data, checkpoints, and the command
  behind every table in the paper
- [`data/stanford_preprocessing/tutorial_Stanford.md`](../data/stanford_preprocessing/tutorial_Stanford.md) —
  the full preprocessing pipeline, from raw download to the `*_features.pkl`
  files this notebook loads
- `python -m unittest discover -s tests` — the release integrity suite
